# **News Artical Classifer Using BERT**

In [44]:
from datasets import load_dataset, load_from_disk
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

import os
from dotenv import load_dotenv

In [45]:
# pip install evaluate

# Download & save to local dataset, tokenizer, model
- Should define exactly the dataset not only "ag_news"

### Hugging face
- For speed and rate let login to hugging face
- line : *https://huggingface.co/settings/tokens*
- create a token and store it in.env file to load

In [46]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [47]:
from huggingface_hub import login
login(hf_token)

In [48]:
# Data set
dataset = load_dataset("SetFit/ag_news")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})


In [49]:
# Downloading models

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [50]:
base_model  = AutoModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Save locally

In [51]:
dataset.save_to_disk("./dataset")
base_model .save_pretrained("./bert_model")
tokenizer.save_pretrained("./bert_model")

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_model/tokenizer_config.json', './bert_model/tokenizer.json')

### Dataset

In [52]:
dataset = load_from_disk("./dataset")

In [53]:
dataset["train"][0]   # First sample

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2,
 'label_text': 'Business'}

In [54]:
train_ds = dataset["train"]
test_ds = dataset["test"]

In [55]:
train_ds # This is conceptually a table (similar to a database table, Excel sheet, or Pandas DataFrame)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 120000
})

In [56]:
train_ds[0] # first row

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2,
 'label_text': 'Business'}

In [57]:
train_ds["text"] # first column

Column(["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.', "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.", 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.', 'Oil prices soar to all-time record, posing new menace to US economy

In [58]:
# model_name = "bert-base-uncased"

# Import the same tokenizer(vocabulary) that bert trained on
# tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained("./bert_model") # Offline

### Let test with single sentence

In [59]:
# sample text to check with tokenizer
sample_text = "The stock market saw record gains today."

"""
convert text into IDs and attention mask, Do several things
Lowercase,
Split into tokens
Add special tokens([CLS] - Beginning & [SEP] - at the end)
    - [CLS] ? Classification token >>> BERT stores the meaning of the entire sentence here
    - [SEP] ? Separator token >>> Used to mark sentence boundaries, Example : [CLS] Sentence A [SEP] Sentence B [SEP]
Convert tokens to IDs ( [CLS] = 101, he   = 1996, stock = 4518, ..., [SEP] = 102 )
[CLS] the stock market saw record gains today [SEP] BECOME [101,1996,4518,3006,2387,2501,12154,2651,102] -> BERT actually sees
Create Attention Mask

Final Output Dictionary "inputs" contins :
{
    'input_ids': tensor(...),
    'attention_mask': tensor(...)
}
input_ids = tensor([[101,1996,4518,3006,2387,2501,12154,2651,102]]), Shape = torch.Size([1,9]) > 1 sentence 9 tokens
attention_mask = tensor([[1,1,1,1,1,1,1,1,1]]),Shape > torch.Size([1,9]) means All tokens are valid (No padding yet tat take all sentence in to
same size by adding 0s to other positions)
token_type_ids  = Which sentence does each token belong to?(since we used 1 senetence all are belongs to it/0
"""

inputs = tokenizer(sample_text, return_tensors="pt") # dictionary
print(f"\n{inputs}\n")

# let show tokenized input
for k, v, in inputs.items():
    print(f"{k} {v.shape} >>> {v}")


{'input_ids': tensor([[  101,  1996,  4518,  3006,  2387,  2501, 12154,  2651,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

input_ids torch.Size([1, 10]) >>> tensor([[  101,  1996,  4518,  3006,  2387,  2501, 12154,  2651,  1012,   102]])
token_type_ids torch.Size([1, 10]) >>> tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask torch.Size([1, 10]) >>> tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [60]:
# Function to tokenize text list from a given table's row

def preprocess_fucntion(examples):
    #print(f"*** {type(examples)}")  # Type is LazyBatch , LazyBatch behaves like a dictionary
    #print(f"*** {type(examples['text'])}") # type is List

    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length"
    )

    # Trainer expect the name xactly labels but here it as lable(no 's') can't be recognized by trainer
    tokens["labels"] = examples["label"]
    return tokens

In [61]:
# Preprocess_function return new dictinarryher map function merge this new dic into existing dic
# preprocess_function returns a dictionary containing new columns (input_ids, token_type_ids, attention_mask)
# map() applies the function to the dataset and merges the returned columns into the existing dataset
# With batched=True, examples["text"] is a list of texts from a batch, not a single text. The tokenizer processes the entire batch at once

train_ds_processed = train_ds.map(preprocess_fucntion, batched=True)
test_ds_processed = test_ds.map(preprocess_fucntion, batched=True)

In [62]:
train_ds_processed

Dataset({
    features: ['text', 'label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 120000
})

In [63]:
train_ds_processed[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2,
 'label_text': 'Business',
 'input_ids': [101,
  2813,
  2358,
  1012,
  6468,
  15020,
  2067,
  2046,
  1996,
  2304,
  1006,
  26665,
  1007,
  26665,
  1011,
  2460,
  1011,
  19041,
  1010,
  2813,
  2395,
  1005,
  1055,
  1040,
  11101,
  2989,
  1032,
  2316,
  1997,
  11087,
  1011,
  22330,
  8713,
  2015,
  1010,
  2024,
  3773,
  2665,
  2153,
  1012,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

### Base/Original model

In [64]:
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"

# model = AutoModel.from_pretrained(model_name)
base_model = AutoModel.from_pretrained("./bert_model") # Offline

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [65]:
print(base_model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

### Model

- BERT is pretrained for general language understanding
- It outputs:[CLS] → 768-dimensional vector (768 hidden features (no classification head yet))
- when we add num_labels=4 >>> Hugging Face automatically replaces the final layer
-
#### What is this "classifier head"?
- It is a simple neural layer: Linear(768 -> num_labels(4))
- The model outputs: logits = [2.1, -0.5, 3.2, 0.7]

#### What happens during training?
- **Step 1**: BERT encodes text
  - "Oil prices rise..." -> 768-d vector
  - **Step 2**: classifier predicts
    - -> 4 logits
  - **Step 3**: softmax
    - softmax(logits)
    - turns it into probabilities > [0.05, 0.10, 0.80, 0.05]
  - **Step 4**: loss calculation
    - If true label = Business (2):
    - model learns:increase class 2 probability, decrease others

In [66]:
# Show what are the lables

for i in np.unique(dataset["train"]["label_text"]):
    print(i)

Business
Sci/Tech
Sports
World


In [67]:
num_labels = len(np.unique(dataset["train"]["label"]))
print(num_labels)

# BERT changes its final layer from : 768 -> hidden representation to 768 -> 4 classification layer that outputs when the model loading

# Online
# classifier_model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     num_labels=num_labels
# )

# Offline
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    "./bert_model",   # local path
    num_labels=num_labels
)

4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ./bert_model
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [68]:
print(classifier_model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Parameters

In [71]:
def count_params(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "Total" : total_params,
        "Trainable" : trainable_params,
        "Frozen" : total_params - trainable_params
    }

In [72]:
print(next(classifier_model.parameters()).requires_grad) ### True means any paramter can be changed

True


#### Freeze the all paramters(can't change)
- requires_grad tells PyTorch whether a parameter should be updated during training
- When we load BERT default, we can change any paramter, check with
- print(next(classifier_model.parameters()).requires_grad) >>> True
- But here we are telling to model > Do not update any weights inside the pretrained BERT model, by setting False

**What gets frozen?**<br>
BertForSequenceClassification(<br>
  (bert): BertModel(...)<br>
  (dropout): Dropout(...)<br>
  (classifier): Linear(768 → 4)<br>
)<br>

The loop freezes everything inside : (bert): BertModel(...), including: embeddings, all 12 transformer layers, pooler

**What still trains?**
- The classifier head: (classifier): Linear(768 → 4), so only this layer learns

In [73]:
for param in classifier_model.base_model.parameters():
    param.requires_grad = False

In [74]:
print(next(classifier_model.parameters()).requires_grad)

False


In [75]:
# Base model
count_params(base_model)

{'Total': 109482240, 'Trainable': 109482240, 'Frozen': 0}

In [76]:
# After freeeze the model check how many params can be changed
count_params(classifier_model) # Trainable parameters are approximately 768 × 4 + 4 = 3,076

{'Total': 109485316, 'Trainable': 3076, 'Frozen': 109482240}

#### What can trainable ?
- The pooler is this part of your model:
   - (pooler): BertPooler(<br>
         (dense): Linear(in_features=768, out_features=768, bias=True)<br>
         (activation): Tanh()<br>
     )<br>
- param.requires_grad = True >>> Allow the pooler weights to be updated during training

**Now**<br>
Embeddings        - Frozen<br>
Encoder Layers    - Frozen<br>
Pooler            - Trainable<br>
Classifier Head   - Trainable<br>

In [77]:
for param in classifier_model.bert.pooler.parameters():
    param.requires_grad = True

In [78]:
count_params(classifier_model)

{'Total': 109485316, 'Trainable': 593668, 'Frozen': 108891648}

#### weights that will change during training

In [79]:
for name, param in classifier_model.named_parameters():
    if param.requires_grad:
        print(name)

bert.pooler.dense.weight
bert.pooler.dense.bias
classifier.weight
classifier.bias


In [80]:
inputs # sample text earlier we used to test

{'input_ids': tensor([[  101,  1996,  4518,  3006,  2387,  2501, 12154,  2651,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [81]:
outputs = classifier_model(**inputs) # ** is called dictionary unpacking
print(outputs) # logits are raw confidence scores for each class

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.4361, -0.0212,  0.6861,  0.4825]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [82]:
# max value
class_pred = outputs.logits.argmax(dim=1)
class_pred

tensor([2])

### Compute matrix function

In [83]:
# Loads the accuracy calculator from Hugging Face evaluate library
# accuracy_metric is now an object that can compute accuracy like: accuracy_metric.compute(...)
accuracy_metric = evaluate.load('accuracy', download_mode="force_redownload")

# Compute metric function, Trainer automatically sends: (eval_logits, eval_labels)
# converts model outputs (logits) into predictions and then calculates how often they match the true labels
def compute_metric(eval_pred):

    # logits=raw model outputs, labels=true correct answers
    logits, labels = eval_pred

    # pick max score
    predictions = np.argmax(logits, axis=1)

    # comparing, predictions(model output) with references(correct labels)
    return accuracy_metric.compute(predictions=predictions, references= labels)

### Training arguments

In [84]:
# training configuration for Hugging Face Trainer
# controls how the model learns, how often it evaluates, and what gets saved

training_args = TrainingArguments(
    output_dir="./result", # Where everything will be saved : trained model, checkpoints, logs
    eval_strategy="epoch", # Evaluate model after each epoch (Epoch 1 -> evaluate, Epoch 2 -> evaluate, ...)
    save_strategy="epoch", # Save model after each epoch (checkpoint-1, checkpoint-2, ...)
    logging_strategy="steps", # Print training info every 10 steps: (step 10 → loss = 1.2, ...)
    logging_steps=10,
    learning_rate=5e-5, # How fast model learns
    per_device_train_batch_size=16, # How many samples processed at once (bigger = faster but more memory)
    per_device_eval_batch_size=16,
    num_train_epochs=3, # Model sees full dataset 3 times
    weight_decay=0.01, # adds small penalty to large weights (Prevents overfitting)
    load_best_model_at_end=True, # After training finishes: automatically loads best-performing model
    metric_for_best_model="accuracy", # Trainer will compare models using accuracy
    report_to="none"
)

In [87]:
from torch.optim import AdamW

# Create a custom optimizer with fused=False
optimizer = AdamW(classifier_model.parameters(), lr=training_args.learning_rate, fused=False)

trainer = Trainer(
    model=classifier_model,
    args=training_args,
    train_dataset=train_ds_processed, # should already tokenized
    eval_dataset=test_ds_processed,
    compute_metrics=compute_metric, # takes logits + labels, converts logits → predictions, computes accuracy
    optimizers=(optimizer, None) # Pass the custom optimizer and no LR scheduler
)

In [88]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.283766,0.314863,0.889342
2,0.332891,0.302090,0.895921
3,0.287974,0.299916,0.895000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=22500, training_loss=0.3374948778523339, metrics={'train_runtime': 1706.9032, 'train_samples_per_second': 210.908, 'train_steps_per_second': 13.182, 'total_flos': 9.472168083456e+16, 'train_loss': 0.3374948778523339, 'epoch': 3.0})

In [89]:
trainer.save_model("/content/my_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Getting correct lable map using exisiting dataset

In [90]:
import pandas as pd

df = pd.DataFrame({
    "label": dataset["train"]["label"],
    "label_text": dataset["train"]["label_text"]
})

print(df.drop_duplicates().sort_values("label"))
label_map = dict(df.drop_duplicates().sort_values("label")[["label", "label_text"]].values)

     label label_text
492      0      World
448      1     Sports
0        2   Business
78       3   Sci/Tech


In [92]:
import torch

def classify(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs= {k:v.to(classifier_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = classifier_model(**inputs)
    predicted_class_id = outputs.logits.argmax(dim=-1).item()

    return label_map[predicted_class_id]

In [93]:
classify("The stock market saw record gains today.")

'Business'